In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from sklearn.model_selection import train_test_split

import pandas as pd
import numpy as np
from typing import Literal

import matplotlib.pyplot as plt

seed = 123

# Data

In [2]:
# display number given a single flatten matrix


def show(matrix: torch.Tensor):
    """Display a number given a single flatten matrix

    Args:
        matrix (torch.Tensor): 1-D array of 784 elements representing a 28x28 image

    Returns:
        plt.imshow object: Display the image
    """
    return plt.imshow(matrix.reshape(28, 28), cmap="gray")

In [3]:
train_path = "../data/train.csv"
test_path = "../data/test.csv"
submission_path = "../data/sample_submission.csv"

In [4]:
class Data:
    """attributes:
    - Train (pd.DataFrame): original train data
    - Test (pd.DataFrame): original test data
    - Submission (pd.DataFrame): original submission data
    - train (pd.DataFrame): train data after splitting
    - cv1 (pd.DataFrame): cv1 data after splitting
    - cv2 (pd.DataFrame): cv2 data after splitting
    - *x_train (torch.Tensor): x data for train
    - *y_train (torch.Tensor): y data for train
    - *x_cv1 (torch.Tensor): x data for cv1
    - *y_cv1 (torch.Tensor): y data for cv1
    - *x_cv2 (torch.Tensor): x data for cv2
    - *y_cv2 (torch.Tensor): y data for cv2
    - *test (torch.Tensor): x data for test
    - nout (int): number of unique outputs
    """

    def __init__(
        self,
        train: str,
        test: str,
        submission: str,
        cv1_size: float = 0.1,
        cv2_size: float = 0.1,
    ) -> None:
        """Data class to load the data and split it into train, test and submission dataframes

        Args:
            train (str): path for train.csv
            test (str): path for test.csv
            submission (str): path for sample_submission.csv
            cv1_size (float, optional): size of cv1 relative to train. Defaults to 0.1.
            cv2_size (float, optional): size of cv2 relative to train. Defaults to 0.1.
        """
        # import original data
        self.Train = pd.read_csv(train)
        self.Test = pd.read_csv(test)
        self.Submission = pd.read_csv(submission)

        # split data into train, cv1, cv2
        self.train, _test = train_test_split(
            self.Train, test_size=cv1_size + cv2_size, random_state=seed, stratify=self.Train["label"]
        )
        self.cv1, self.cv2 = train_test_split(
            _test, test_size=cv2_size / (cv1_size + cv2_size), random_state=seed, stratify=_test["label"]
        )
        del _test

        # split data into X, y
        self.x_train = torch.tensor(self.train.drop(columns=["label"]).values, dtype=torch.float32)
        self.y_train = torch.tensor(self.train["label"].values)
        self.x_cv1 = torch.tensor(self.cv1.drop(columns=["label"]).values, dtype=torch.float32)
        self.y_cv1 = torch.tensor(self.cv1["label"].values)
        self.x_cv2 = torch.tensor(self.cv2.drop(columns=["label"]).values, dtype=torch.float32)
        self.y_cv2 = torch.tensor(self.cv2["label"].values)
        self.test = torch.tensor(self.Test.values, dtype=torch.float32)

        # unique output
        self.nout = len(self.Train["label"].unique())

In [5]:
d = Data(
    train_path,
    test_path,
    submission_path,
    cv1_size=0.1,
    cv2_size=0.1,
)

# Test

In [133]:
c = nn.Conv2d(
    in_channels=1,
    out_channels=4,
    kernel_size=3,
    padding=1
)
p = nn.MaxPool2d(2)
a = d.x_train[0].reshape(1, 1, 28, 28)
c(a).shape # torch.Size([1, 4, 28, 28])

torch.Size([1, 4, 28, 28])

In [1]:
def calculate_conv2d_shape(input_shape, kernel_size, stride=1, padding=0, dilation=1):
    """
    Calculate the input and output shape for a Conv2D layer.

    Args:
        input_shape (tuple): Shape of the input tensor (channels, height, width).
        kernel_size (int or tuple): Size of the convolutional kernel.
        stride (int or tuple, optional): Stride of the convolution. Default is 1.
        padding (int or tuple, optional): Padding added to the input. Default is 0.
        dilation (int or tuple, optional): Spacing between kernel elements. Default is 1.

    Returns:
        tuple: Input shape and output shape of the Conv2D layer.
    """
    input_channels, input_height, input_width = input_shape

    if isinstance(kernel_size, int):
        kernel_size = (kernel_size, kernel_size)

    if isinstance(stride, int):
        stride = (stride, stride)

    if isinstance(padding, int):
        padding = (padding, padding)

    if isinstance(dilation, int):
        dilation = (dilation, dilation)

    # Calculate output shape
    output_channels = input_channels
    output_height = ((input_height + 2 * padding[0] - dilation[0] * (kernel_size[0] - 1) - 1) // stride[0]) + 1
    output_width = ((input_width + 2 * padding[1] - dilation[1] * (kernel_size[1] - 1) - 1) // stride[1]) + 1

    input_shape = (input_channels, input_height, input_width)
    output_shape = (output_channels, output_height, output_width)

    return input_shape, output_shape


In [9]:
calculate_conv2d_shape(
    input_shape=(1, 28, 28),
    kernel_size=3,
    stride=3,
    padding=1,
    dilation=1,
)

((1, 28, 28), (1, 10, 10))